# Notebook 01: Universe Selection

## European LSTM Portfolio - Stock Universe Construction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/european_lstm_project/notebooks/01_universe_selection.ipynb)

---

## 📚 Academic Objective

Construct a balanced universe of 44-50 liquid European stocks using **stratified sector sampling**.

### Methodology:
1. Query WRDS Compustat Global for European stocks
2. Apply liquidity filters (Amihud, 2002):
   - Market cap > €2 Billion
   - Avg daily volume > €1 Million
   - Continuous 5-year history
3. Stratified sampling: 4 stocks per GICS sector (Fama-French, 1993)
4. Download benchmark (STOXX Europe 600)

### Expected Output:
- `data/processed/universe_tickers.csv` - 44 selected stocks
- `data/raw/stock_prices.parquet` - Historical price/volume data
- `data/market_data/stoxx600.csv` - Benchmark data

---

## ⏱️ Estimated Runtime:
- **Google Colab**: 5-10 minutes (depends on WRDS connection speed)
- **Local**: 5-10 minutes

---

## 🔧 Step 1: Environment Setup

In [ ]:
# Detect environment and setup paths
import sys
import os

# Check if running on Google Colab
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    print("🔵 Running on Google Colab")
    
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Set base path
    BASE_PATH = '/content/drive/MyDrive/european_lstm_project'
    
    # Navigate to project directory
    os.chdir(BASE_PATH)
    
    # Install requirements
    !pip install -q wrds yfinance pandas numpy
    
else:
    print("💻 Running locally")
    BASE_PATH = os.path.abspath('..')
    os.chdir(BASE_PATH)

# Add src to Python path
sys.path.insert(0, os.path.join(BASE_PATH, 'src'))

print(f"✓ Base directory: {BASE_PATH}")
print(f"✓ Current working directory: {os.getcwd()}")

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Import project modules
import config
from data_loader import WRDSDataLoader, BenchmarkDataLoader, filter_liquid_stocks, stratified_sector_selection

# Display configuration
config.print_config_summary()

## 📊 Step 2: Connect to WRDS and Query European Stocks

### Academic Note:
We use **WRDS Compustat Global** which includes:
- Fundamental data (balance sheet, income statement)
- Daily prices and volumes
- Corporate actions (splits, dividends)
- Includes delisted stocks (avoids survivorship bias - Brown et al., 1995)

In [ ]:
# Initialize WRDS connection
wrds_loader = WRDSDataLoader()

# Connect (will prompt for credentials)
print("Connecting to WRDS...")
print("You will be prompted for your WRDS credentials.")
print("Password input will be hidden for security.")
print("-" * 60)

wrds_loader.connect()

In [ ]:
# Query European stocks
print("Querying Compustat Global for European stocks...")
print(f"Date range: {config.DATA_START_DATE} to {config.DATA_END_DATE}")
print(f"Countries: {len(config.EUROPEAN_COUNTRY_CODES)} European countries")
print("-" * 60)

european_stocks = wrds_loader.get_european_universe(
    country_codes=config.EUROPEAN_COUNTRY_CODES,
    start_date=config.DATA_START_DATE,
    end_date=config.DATA_END_DATE,
    min_market_cap=config.MIN_MARKET_CAP,
    min_avg_volume=config.MIN_AVG_VOLUME
)

print(f"✓ Retrieved {len(european_stocks)} records")
print(f"✓ Unique stocks: {european_stocks['gvkey'].nunique()}")
print(f"✓ Date range: {european_stocks['datadate'].min()} to {european_stocks['datadate'].max()}")

# Display sample
european_stocks.head()

## 🔍 Step 3: Apply Liquidity Filters

### Academic Justification (Amihud, 2002):

**Amihud (2002)** shows that illiquidity is priced in equity returns:
- Illiquid stocks have higher expected returns (liquidity premium)
- But high transaction costs make them unsuitable for active strategies

**Our filters:**
1. **Market Cap > €2B**: Ensures institutional-quality stocks
2. **Daily Volume > €1M**: Enables efficient execution
3. **5-year history**: Sufficient data for statistical estimation

In [ ]:
# Apply liquidity filters
print("Applying liquidity filters...")
print(f"Criteria:")
print(f"  - Market Cap > €{config.MIN_MARKET_CAP:,.0f}")
print(f"  - Avg Daily Volume > €{config.MIN_AVG_VOLUME:,.0f}")
print(f"  - Min history: {config.MIN_HISTORY_YEARS} years (~1260 trading days)")
print("-" * 60)

liquid_stocks = filter_liquid_stocks(
    df=european_stocks,
    min_market_cap=config.MIN_MARKET_CAP,
    min_avg_volume=config.MIN_AVG_VOLUME,
    min_history_days=config.MIN_HISTORY_YEARS * 252
)

print(f"\n✓ Filtered to {liquid_stocks['gvkey'].nunique()} liquid stocks")

In [ ]:
# Analyze sector distribution
latest_data = liquid_stocks.sort_values('datadate').groupby('gvkey').last()

print("Sector Distribution (before selection):")
print("-" * 60)
sector_counts = latest_data['gics_sector'].value_counts()
print(sector_counts)

# Visualize
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
sector_counts.plot(kind='barh')
plt.title('Number of Liquid Stocks per GICS Sector')
plt.xlabel('Count')
plt.ylabel('Sector')
plt.tight_layout()
plt.show()

## 🎯 Step 4: Stratified Sector Selection

### Academic Justification (Fama-French, 1993):

**Fama & French (1993)** show that:
- Industry (sector) is a significant factor in returns
- Sector diversification reduces idiosyncratic risk

**Our approach:**
- Select top 4 stocks per sector by market cap
- 11 GICS sectors × 4 stocks = **44 total stocks**
- Ensures balanced exposure (no sector > 15%)

In [ ]:
# Stratified selection
print("Performing stratified sector selection...")
print(f"Target: {config.STOCKS_PER_SECTOR} stocks per sector")
print("-" * 60)

selected_gvkeys = stratified_sector_selection(
    df=liquid_stocks,
    sectors=config.GICS_SECTORS,
    stocks_per_sector=config.STOCKS_PER_SECTOR
)

print(f"\n✓ Final universe: {len(selected_gvkeys)} stocks")

In [ ]:
# Create final universe DataFrame
universe_df = latest_data[latest_data.index.isin(selected_gvkeys)].copy()

# Add metadata
universe_df = universe_df[[
    'company_name', 'country_code', 'gics_sector', 'market_cap', 'volume_value'
]].reset_index()

# Sort by sector and market cap
universe_df = universe_df.sort_values(['gics_sector', 'market_cap'], ascending=[True, False])

print("\n📋 Final Universe Composition:")
print("=" * 80)
print(universe_df.to_string(index=False))
print("=" * 80)

# Summary statistics
print("\n📊 Universe Statistics:")
print(f"Total stocks: {len(universe_df)}")
print(f"Total market cap: €{universe_df['market_cap'].sum()/1e9:.1f}B")
print(f"Median market cap: €{universe_df['market_cap'].median()/1e9:.1f}B")
print(f"Countries represented: {universe_df['country_code'].nunique()}")

In [ ]:
# Verify sector balance
sector_distribution = universe_df['gics_sector'].value_counts()

print("\n✓ Sector Balance Check:")
print("-" * 60)
for sector, count in sector_distribution.items():
    pct = count / len(universe_df) * 100
    status = "✓" if count == config.STOCKS_PER_SECTOR else "⚠"
    print(f"{status} {sector:30s}: {count} stocks ({pct:.1f}%)")

# Check if any sector exceeds limit
max_sector_pct = sector_distribution.max() / len(universe_df)
if max_sector_pct <= config.MAX_SECTOR_WEIGHT:
    print(f"\n✓ All sectors within {config.MAX_SECTOR_WEIGHT*100}% limit")
else:
    print(f"\n⚠ Warning: Some sectors exceed {config.MAX_SECTOR_WEIGHT*100}% limit")

## 💾 Step 5: Download Historical Data

In [ ]:
# Filter full dataset to selected stocks
historical_data = liquid_stocks[liquid_stocks['gvkey'].isin(selected_gvkeys)].copy()

print(f"Historical data shape: {historical_data.shape}")
print(f"Date range: {historical_data['datadate'].min()} to {historical_data['datadate'].max()}")
print(f"Total records: {len(historical_data):,}")

In [ ]:
# Download fundamentals for selected stocks
print("\nDownloading fundamental data...")

fundamentals = wrds_loader.get_fundamentals(
    gvkeys=selected_gvkeys,
    start_date=config.DATA_START_DATE,
    end_date=config.DATA_END_DATE
)

print(f"✓ Retrieved {len(fundamentals)} fundamental records")
fundamentals.head()

## 📈 Step 6: Download Benchmark (STOXX Europe 600)

In [ ]:
# Download benchmark
print("Downloading benchmark index (STOXX Europe 600)...")

benchmark = BenchmarkDataLoader.download_benchmark(
    ticker=config.BENCHMARK_TICKER,
    start_date=config.DATA_START_DATE,
    end_date=config.DATA_END_DATE
)

print(f"✓ Downloaded {len(benchmark)} days of benchmark data")

# Plot benchmark
plt.figure(figsize=(12, 6))
plt.plot(benchmark.index, benchmark['Close'])
plt.title('STOXX Europe 600 - Historical Performance')
plt.xlabel('Date')
plt.ylabel('Price')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate cumulative return
cum_return = (benchmark['Close'].iloc[-1] / benchmark['Close'].iloc[0] - 1) * 100
print(f"\n📊 Benchmark cumulative return: {cum_return:.2f}%")

## 💾 Step 7: Save Outputs

In [ ]:
# Create output directories
os.makedirs(config.PROCESSED_DATA_DIR, exist_ok=True)
os.makedirs(config.RAW_DATA_DIR, exist_ok=True)
os.makedirs(config.MARKET_DATA_DIR, exist_ok=True)

print("Saving outputs...")
print("-" * 60)

# 1. Universe ticker list
universe_file = os.path.join(config.PROCESSED_DATA_DIR, 'universe_tickers.csv')
universe_df.to_csv(universe_file, index=False)
print(f"✓ Saved: {universe_file}")

# 2. Historical price/volume data
prices_file = os.path.join(config.RAW_DATA_DIR, 'stock_prices.parquet')
historical_data.to_parquet(prices_file, index=False)
print(f"✓ Saved: {prices_file}")

# 3. Fundamental data
fundamentals_file = os.path.join(config.RAW_DATA_DIR, 'fundamentals.parquet')
fundamentals.to_parquet(fundamentals_file, index=False)
print(f"✓ Saved: {fundamentals_file}")

# 4. Benchmark data
benchmark_file = os.path.join(config.MARKET_DATA_DIR, 'stoxx600.csv')
benchmark.to_csv(benchmark_file)
print(f"✓ Saved: {benchmark_file}")

print("\n✅ All outputs saved successfully!")

## 🔐 Step 8: Disconnect from WRDS

In [ ]:
# Disconnect
wrds_loader.disconnect()
print("✓ Disconnected from WRDS")

## ✅ Summary & Next Steps

### What We Accomplished:
1. ✅ Connected to WRDS Compustat Global
2. ✅ Queried European stocks (20 countries)
3. ✅ Applied liquidity filters (market cap, volume, history)
4. ✅ Stratified sector selection (4 stocks × 11 sectors)
5. ✅ Downloaded historical prices, volumes, fundamentals
6. ✅ Downloaded STOXX 600 benchmark
7. ✅ Saved all data for next notebook

### Outputs Created:
- `data/processed/universe_tickers.csv` - **44 selected stocks**
- `data/raw/stock_prices.parquet` - Price/volume history
- `data/raw/fundamentals.parquet` - Fundamental ratios
- `data/market_data/stoxx600.csv` - Benchmark index

### Academic Validation:
- ✅ **Zero survivorship bias** (WRDS includes delisted stocks)
- ✅ **Sector diversification** (Fama-French, 1993)
- ✅ **Liquidity screening** (Amihud, 2002)
- ✅ **Institutional-quality universe** (€2B+ market cap)

---

## ➡️ Next Notebook: 02_feature_engineering.ipynb

In the next notebook, we will:
1. Calculate rolling beta vs. STOXX 600
2. Compute residual returns (alpha signal)
3. Generate 20+ technical indicators
4. Download macro variables (oil, rates, volatility, FX)
5. Apply PCA dimensionality reduction
6. Create binary target variable for LSTM training

---

**📖 References:**
- Amihud, Y. (2002). "Illiquidity and stock returns." *Journal of Financial Markets*, 5(1), 31-56.
- Brown et al. (1995). "Survival." *Journal of Finance*, 50(3), 853-873.
- Fama, E. F., & French, K. R. (1993). "Common risk factors." *Journal of Financial Economics*, 33(1), 3-56.